In [1]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

모델: gpt-5.6-luna


In [10]:
from neo4j import GraphDatabase

# os.getenv 의 두 번째 인자가 기본값이다. .env 에 값이 없으면 이 값으로 접속한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 연결이 안 되면 여기서 바로 에러가 난다. 뒤 셀까지 가지 않는다


def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # 값은 query 에 붙이지 않고 params 로 따로 넘긴다. 따옴표가 든 근거 문장도 안전하다
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


# 레시피 지식그래프 온톨로지
앞 3,000건을 대상으로 재료 수량·주재료/부재료·시간·도구·난이도·양을 모델링한다. 추천의 최우선 규칙은 보유량이 요구량보다 적은 레시피를 제외하는 것이다.

In [5]:
import json, os, re
from pathlib import Path
DATA_PATH=next((p for p in [Path.cwd()/'전처리/recipes_10000_preprocessed.jsonl',Path.cwd()/'recipes_10000_preprocessed.jsonl',Path.cwd().parent/'전처리/recipes_10000_preprocessed.jsonl'] if p.exists()),None)
assert DATA_PATH, 'recipes_10000_preprocessed.jsonl을 찾지 못했습니다.'
recipes=[]
with DATA_PATH.open(encoding='utf-8') as f:
    for seq,line in enumerate(f):
        if seq>=3000: break
        if line.strip():
            x=json.loads(line); x['_sequence']=seq+1; recipes.append(x)
print(DATA_PATH,len(recipes))

c:\Users\Playdata\Desktop\mle-01-p2-team2\전처리\recipes_10000_preprocessed.jsonl 3000


## 스키마
노드: `Recipe`, `Ingredient`, `Component`, `Tool`, `Difficulty`, `TimeBand`, `ServingBand`.
관계: `Recipe-[:HAS_COMPONENT]->Component-[:USES_INGREDIENT]->Ingredient`, `Component-[:IS_MAIN|IS_SEASONING]->Ingredient`, `Recipe-[:REQUIRES_TOOL|HAS_DIFFICULTY|HAS_TIME_BAND|HAS_SERVING_BAND]->...`.
Component는 양·단위·최소/최대량·수량 파싱 상태를 보존하는 관계 객체다. 같은 Ingredient를 공유하므로 같은 재료를 쓰는 레시피를 연결할 수 있다.

In [6]:
NODE_SCHEMA={'Recipe':['recipe_uid','sequence','title','source_url','servings_raw','servings_min','servings_max','cooking_time_raw','time_min','time_max','difficulty'],'Ingredient':['name','name_normalized'],'Component':['component_uid','index','role','group','is_main','is_required','amount_raw','quantity_min','quantity_max','unit','quantity_parse_status','preparation'],'Tool':['name'],'Difficulty':['name','rank'],'TimeBand':['name','min_minutes','max_minutes'],'ServingBand':['name','min_servings','max_servings']}
RELATION_SCHEMA={'HAS_COMPONENT':'Recipe->Component','USES_INGREDIENT':'Component->Ingredient','IS_MAIN':'Component->Ingredient','IS_SEASONING':'Component->Ingredient','REQUIRES_TOOL':'Recipe->Tool','HAS_DIFFICULTY':'Recipe->Difficulty','HAS_TIME_BAND':'Recipe->TimeBand','HAS_SERVING_BAND':'Recipe->ServingBand','SHARES_INGREDIENT':'Recipe->Recipe'}
print(NODE_SCHEMA,RELATION_SCHEMA)

{'Recipe': ['recipe_uid', 'sequence', 'title', 'source_url', 'servings_raw', 'servings_min', 'servings_max', 'cooking_time_raw', 'time_min', 'time_max', 'difficulty'], 'Ingredient': ['name', 'name_normalized'], 'Component': ['component_uid', 'index', 'role', 'group', 'is_main', 'is_required', 'amount_raw', 'quantity_min', 'quantity_max', 'unit', 'quantity_parse_status', 'preparation'], 'Tool': ['name'], 'Difficulty': ['name', 'rank'], 'TimeBand': ['name', 'min_minutes', 'max_minutes'], 'ServingBand': ['name', 'min_servings', 'max_servings']} {'HAS_COMPONENT': 'Recipe->Component', 'USES_INGREDIENT': 'Component->Ingredient', 'IS_MAIN': 'Component->Ingredient', 'IS_SEASONING': 'Component->Ingredient', 'REQUIRES_TOOL': 'Recipe->Tool', 'HAS_DIFFICULTY': 'Recipe->Difficulty', 'HAS_TIME_BAND': 'Recipe->TimeBand', 'HAS_SERVING_BAND': 'Recipe->ServingBand', 'SHARES_INGREDIENT': 'Recipe->Recipe'}


In [7]:
def qvals(q):
    q=q or {}; return (q.get('min') if q.get('min') is not None else q.get('value'),q.get('max') if q.get('max') is not None else q.get('value'),q.get('unit'),q.get('type'),q.get('parse_status'))
TIME_BANDS=[('초간단',0,15),('간단',16,30),('보통',31,60),('오래 걸림',61,10**9)]; SERVING_BANDS=[('적은 양',0,2),('보통 양',3,5),('많은 양',6,10**9)]; DIFFICULTY_RANK={'아무나':0,'초급':1,'중급':2,'고급':3}
def label(v,b):
    return next((n for n,lo,hi in b if v is not None and lo<=v<=hi),'미상')
def normalize(r):
    s=r.get('servings') or {}; t=r.get('cooking_time') or {}; smn,smx,_,_,_=qvals(s); tmn,tmx,_,_,_=qvals(t); comps=[]
    for c in r.get('components',[]):
        if c.get('role') not in ('ingredient','seasoning'): continue
        q=c.get('quantity') or {}; lo,hi,u,qt,ps=qvals(q); name=c.get('base_name') or c.get('name_normalized') or c.get('name_raw') or '미상 재료'
        comps.append({'component_uid':f"{r['recipe_uid']}:{c.get('index',0)}",'index':c.get('index'),'role':c.get('role'),'group':c.get('group'),'is_main':c.get('role')=='ingredient','is_required':True,'name_raw':c.get('name_raw'),'amount_raw':q.get('raw') or c.get('amount_raw'),'quantity_type':qt,'quantity_min':lo,'quantity_max':hi,'unit':u,'quantity_parse_status':ps,'preparation':c.get('preparation'),'ingredient_name':name})
    return {'recipe_uid':r.get('recipe_uid'),'sequence':r['_sequence'],'title':r.get('title'),'description':r.get('description'),'source_url':r.get('source_url'),'servings_raw':s.get('raw'),'servings_min':smn,'servings_max':smx,'cooking_time_raw':t.get('raw'),'time_min':tmn,'time_max':tmx,'difficulty':r.get('difficulty') or '미상','time_band':label(tmn,TIME_BANDS),'serving_band':label(smn,SERVING_BANDS),'components':comps,'tools':r.get('tools') or []}
graph_records=[normalize(r) for r in recipes]; print('components',sum(len(r['components']) for r in graph_records))

components 29003


In [8]:
from neo4j import GraphDatabase
driver=GraphDatabase.driver(os.getenv('NEO4J_URI','bolt://localhost:7687'),auth=(os.getenv('NEO4J_USER','neo4j'),os.getenv('NEO4J_PASSWORD','neo4j')))
CONSTRAINTS=['CREATE CONSTRAINT recipe_uid IF NOT EXISTS FOR (n:Recipe) REQUIRE n.recipe_uid IS UNIQUE','CREATE CONSTRAINT ingredient_name IF NOT EXISTS FOR (n:Ingredient) REQUIRE n.name_normalized IS UNIQUE','CREATE CONSTRAINT component_uid IF NOT EXISTS FOR (n:Component) REQUIRE n.component_uid IS UNIQUE','CREATE CONSTRAINT tool_name IF NOT EXISTS FOR (n:Tool) REQUIRE n.name IS UNIQUE']
def load_graph(records):
  with driver.session() as s:
    for q in CONSTRAINTS:s.run(q).consume()
    for r in records:
      props={k:r[k] for k in ['sequence','title','description','source_url','servings_raw','servings_min','servings_max','cooking_time_raw','time_min','time_max','difficulty']}; s.run('MERGE (r:Recipe {recipe_uid:$id}) SET r += $p',id=r['recipe_uid'],p=props).consume()
      for c in r['components']:
        cp={k:v for k,v in c.items() if k!='ingredient_name'}; s.run('MATCH (r:Recipe {recipe_uid:$id}) MERGE (c:Component {component_uid:$cid}) SET c += $cp MERGE (r)-[:HAS_COMPONENT]->(c) MERGE (i:Ingredient {name_normalized:$n}) SET i.name=$n MERGE (c)-[:USES_INGREDIENT]->(i)',id=r['recipe_uid'],cid=c['component_uid'],cp=cp,n=c['ingredient_name']).consume()
  return 'loaded'
# 실제 적재: load_graph(graph_records)

In [9]:
def recommend(leftovers,records=graph_records,top_k=20):
  have={str(k).strip().lower():v for k,v in leftovers.items()}; out=[]
  for r in records:
    missing=[]; matched=0; warnings=[]
    for c in r['components']:
      n=c['ingredient_name'].strip().lower(); h=have.get(n)
      if not h: missing.append(n); continue
      if c['quantity_min'] is None: warnings.append(n+' 수량 미상'); continue
      if c['unit']!=h.get('unit'): warnings.append(n+' 단위 비교 불가'); continue
      if h.get('amount',0)<c['quantity_min']: missing.append(f"{n} ({h.get('amount')} < {c['quantity_min']}{c['unit']})")
      else: matched+=1
    if not missing: out.append({'recipe_uid':r['recipe_uid'],'sequence':r['sequence'],'title':r['title'],'matched_count':matched,'warnings':warnings,'difficulty':r['difficulty'],'time_band':r['time_band'],'serving_band':r['serving_band'],'source_url':r['source_url']})
  return sorted(out,key=lambda x:(-x['matched_count'],DIFFICULTY_RANK.get(x['difficulty'],99),x['sequence']))[:top_k]
# 예: recommend({'양파':{'amount':2,'unit':'개'}})